In [1]:
import spacy
import pandas as pd
import numpy as np

In [2]:
nlp = spacy.load("pt_core_news_sm")

c:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\.venv\Lib\site-packages\spacy\util.py:971: UserWarning: [W095] Model 'pt_core_news_sm' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [3]:
df = pd.read_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_enriquecido_10062026_v2.parquet")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8515 entries, 0 to 8514
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   TEMA_ORIGEM            8515 non-null   string 
 1   TEMA_CODIGO            8515 non-null   string 
 2   TOTAL_PROC_VINCULADOS  8277 non-null   float64
 3   ID_PROC                8515 non-null   int64  
 4   ques_direito           8277 non-null   str    
 5   tese_firmada           5768 non-null   str    
 6   info_legislativa       3307 non-null   str    
 7   ementa                 8019 non-null   str    
 8   texto                  8140 non-null   str    
 9   inteiro_teor           8515 non-null   string 
dtypes: float64(1), int64(1), str(5), string(3)
memory usage: 2.5 GB


In [32]:
import pandas as pd


def lematizar_texto_longo(texto, nlp_model, max_chunk_size=50000):
    if not isinstance(texto, str) or not texto.strip():
        return ""

    # 1. Quebra o texto em parágrafos para não cortar frases ao meio
    paragrafos = texto.split("\n")
    chunks = []
    current_chunk = []
    current_length = 0

    for para in paragrafos:
        # Se o parágrafo sozinho for maior que o limite, quebra por espaço
        if len(para) > max_chunk_size:
            palavras = para.split(" ")
            for palavra in palavras:
                if current_length + len(palavra) + 1 > max_chunk_size:
                    chunks.append(" ".join(current_chunk))
                    current_chunk = [palavra]
                    current_length = len(palavra)
                else:
                    current_chunk.append(palavra)
                    current_length += len(palavra) + 1
        # Caso contrário, agrupa os parágrafos até atingir o limite do bloco
        elif current_length + len(para) + 1 > max_chunk_size:
            chunks.append("\n".join(current_chunk))
            current_chunk = [para]
            current_length = len(para)
        else:
            current_chunk.append(para)
            current_length += len(para) + 1

    if current_chunk:
        chunks.append("\n".join(current_chunk))

    # 2. Processa cada bloco menor usando o nlp.pipe para máxima eficiência
    componentes_inuteis = []
    lemmas_totais = []

    for doc in nlp_model.pipe(chunks, disable=componentes_inuteis, batch_size=20):
        lemmas_totais.extend([token.lemma_ for token in doc])

    return " ".join(lemmas_totais)


# --- COMO APLICAR NO DATAFRAME ---

# Aplicando a função linha por linha (mas processando internamente em blocos seguros)
df["inteiro_teor_lematizado"] = df["inteiro_teor"].apply(
    lambda x: lematizar_texto_longo(x, nlp)
)

In [36]:
df_lematizado = df[["TEMA_ORIGEM", "TEMA_CODIGO", "inteiro_teor", "inteiro_teor_lematizado"]]
df_lematizado.to_parquet(r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_enriquecido_10062026_lematizado.parquet", index=False)

In [42]:
# Pega a primeira linha válida para comparar
exemplo_original = df_lematizado["inteiro_teor"].iloc[2]
exemplo_lematizado = df_lematizado["inteiro_teor_lematizado"].iloc[2]

print("--- ORIGINAL ---")
print(exemplo_original[:500])  # Mostra os primeiros 500 caracteres

print("\n--- LEMATIZADO ---")
print(exemplo_lematizado[:500])

--- ORIGINAL ---
Erro Parser>>>>>inicio<<<<<
SÚMULAS  As seções de direito penal e direito privado do Superior Tribunal de Justiça (STJ) aprovaram, cada  uma, duas novas súmulas na última quarta-feira (11). Houve também o cancelamento da Súmula  469 pela Segunda Seção.  Os enunciados sumulares são o resumo de entendimentos consolidados nos julgamentos do  tribunal e servem de orientação a toda a comunidade jurídica sobre a jurisprudência do STJ.  Direito penal  A Terceira Seção aprovou os enunciados de número 60

--- LEMATIZADO ---
Erro Parser>>>>>inicio < < < < < 
 SÚMULAS   o seção de direito penal e direito privado de o Superior Tribunal de Justiça ( STJ ) aprovar , cada   um , dois novo súmula em o último quarta-feira ( 11 ) . Houve também o cancelamento de o Súmula   469 por o Segunda Seção .   o enunciado sumular ser o resumo de entendimento consolidar em o julgamento de o   tribunal e servir de orientação a todo o comunidade jurídico sobre o jurisprudência de o STJ .   direito p

In [ ]:
import re
import pandas as pd


def converter_expressao_para_regex(expressao_booleana):
    """Conversor automático de expressões (A OU B) E (C OU D) para Regex Lookahead"""
    if not isinstance(expressao_booleana, str) or not expressao_booleana.strip():
        return ""

    # 1. Encontra tudo o que está dentro dos parênteses (...)
    # Isso isola os grupos que estão separados por " E "
    grupos_parenteses = re.findall(r"\((.*?)\)", expressao_booleana)

    # Se a estrutura não tiver parênteses, tenta quebrar direto pelo " E "
    if not grupos_parenteses:
        grupos_parenteses = expressao_booleana.split(" E ")

    lookaheads = []

    for grupo in grupos_parenteses:
        # Quebra os termos internos pelo " OU "
        termos = [termo.strip() for termo in grupo.split(" OU ") if termo.strip()]

        if not termos:
            continue

        # Ordena do maior termo para o menor (evita que 'DP' engula 'Defensoria Pública')
        termos_ordenados = sorted(termos, key=len, reverse=True)

        # Escapa caracteres especiais que possam existir no texto (como pontos ou interrogações)
        termos_escapados = "|".join(
            [re.escape(termo) for termo in termos_ordenados]
        )

        # Monta o Lookahead para este grupo específico garantindo as fronteiras de palavra (\b)
        # O .*? garante que ele ache o termo em qualquer parte do texto longo
        lookahead_grupo = f"(?=.*?\\b({termos_escapados})\\b)"
        lookaheads.append(lookahead_grupo)

    # Une todos os lookaheads. Para a Regex dar MATCH, TODOS os grupos precisam ser verdadeiros (Lógica "E")
    regex_final = "".join(lookaheads)
    return regex_final


Sucesso! O arquivo 'Temas_STF_com_Regex.csv' foi gerado com as Regex.


In [47]:
# 1. Carrega o seu CSV (ajustando para o encoding padrão que costuma vir do Excel)
df_temas = pd.read_csv(
    r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\Palavras chaves temas do STF - Leonardo.csv", encoding="utf-8"
)
df_temas_2 = pd.read_csv(
    r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\Palavras chaves temas do STJ - Patrícia.csv", encoding="utf-8"
)

# 2. Aplica a conversão automática criando a nova coluna 'REGEX'
df_temas["REGEX"] = df_temas["PALAVRAS-CHAVES"].apply(
    converter_expressao_para_regex
)
df_temas_2["REGEX"] = df_temas_2["PALAVRAS-CHAVES"].apply(
    converter_expressao_para_regex
)
df_temas = pd.concat([df_temas, df_temas_2], ignore_index=True)
# 3. Salva o resultado em um novo arquivo CSV para você usar
df_temas.to_csv("Temas_STF_com_Regex.csv", index=False, encoding="utf-8")

print("Sucesso! O arquivo 'Temas_STF_com_Regex.csv' foi gerado com as Regex.")

Sucesso! O arquivo 'Temas_STF_com_Regex.csv' foi gerado com as Regex.


In [49]:
def recomendar_tema_para_processo(texto_processo, df_regex):
    """
    Função que recomenda o tema mais provável para um processo com base no inteiro teor.
    Retorna o código do tema e a expressão booleana correspondente.
    """
    for _, row in df_regex.iterrows():
        regex = row["REGEX"]
        if re.search(regex, texto_processo, flags=re.IGNORECASE):
            return row["TEMA CÓDIGO"], row["PALAVRAS-CHAVES"]
    return None, None  # Retorna None se nenhum tema for encontrado

In [54]:
import re


def aplicar_regex_no_dataframe(df, df_regex):
    """Aplica as expressões regulares de df_regex no df e lista os temas encontrados."""
    # Inicializa a coluna com listas vazias para cada linha
    df["temas_encontrados"] = [[] for _ in range(len(df))]

    for _, row in df_regex.iterrows():
        regex = row["REGEX"]
        tema_codigo = row["TEMA CÓDIGO"]  # Nome exato da coluna do seu CSV

        # Pula se a regex estiver vazia
        if not regex:
            continue

        # Cria uma máscara booleana para as linhas que dão match
        matches = df["inteiro_teor_lematizado"].str.contains(
            regex, flags=re.IGNORECASE, regex=True, na=False
        )

        # Para as linhas onde deu Match, adiciona o código do tema na lista
        df.loc[matches, "temas_encontrados"] = df.loc[
            matches, "temas_encontrados"
        ].apply(lambda x: x + [tema_codigo])

    return df

In [ ]:
df_lematizado_regex = df_lematizado['inteiro_teor_lematizado'].apply(lambda x: aplicar_regex_no_dataframe(df_lematizado, df_temas))

C:\Users\lfmelo\AppData\Local\Temp\ipykernel_1952\2540749458.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df["inteiro_teor_lematizado"].str.contains(


In [ ]:
df_lematizado_regex.to_parquet(
    r"C:\Users\lfmelo\Documents\Github\TJGO_ThemeClassification\data\dataset_enriquecido_10062026_lematizado_com_regex.parquet", index=False
)